In [24]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv


load_dotenv()

DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT")
DB_NAME = os.getenv("POSTGRES_DB")

engine = create_engine(

   f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
query = "SELECT * FROM  staging.stg_annonces;"
df = pd.read_sql(query, engine)

print(df.head())

  annonce_id date_publication                         titre    ville  \
0  ANO000343       2023-05-05        Terrain moderne Meknès   Meknès   
1  ANO001296       2023-08-20       Terrain moderne Kenitra  Kenitra   
2  ANO001212       2023-06-20     Appartement moderne Oujda    Oujda   
3  ANO001000       2024-01-21  Appartement à vente - Tanger   Tanger   
4  ANO001123       2024-03-19              Beau Villa Rabat    Rabat   

   quartier    type_bien transaction       prix  surface  nb_chambres  \
0    Hamria      Terrain       Vente  811401.04   314.40          0.0   
1    Centre      Terrain    Location    3843.55    36.46          0.0   
2  Hay Qods  Appartement    Location    9190.70    95.54          2.0   
3       NaN  Appartement       Vente   30000.00    93.35          1.0   
4   Souissi        Villa    Location   20844.72   264.22          3.0   

   nb_salles_bain  etage  annee_construction  
0             NaN    0.0              2014.0  
1             0.0    NaN          

In [25]:
print(df.shape)
print(df.info())

(1508, 13)
<class 'pandas.DataFrame'>
RangeIndex: 1508 entries, 0 to 1507
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   annonce_id          1508 non-null   str    
 1   date_publication    1432 non-null   str    
 2   titre               1508 non-null   str    
 3   ville               1508 non-null   str    
 4   quartier            1093 non-null   str    
 5   type_bien           1470 non-null   str    
 6   transaction         1470 non-null   str    
 7   prix                1508 non-null   float64
 8   surface             1508 non-null   float64
 9   nb_chambres         1379 non-null   float64
 10  nb_salles_bain      1404 non-null   float64
 11  etage               1276 non-null   float64
 12  annee_construction  1304 non-null   float64
dtypes: float64(6), str(7)
memory usage: 153.3 KB
None


## Suppression des doublons

In [26]:
df = df.drop_duplicates(subset=["annonce_id"])
print(df.shape)

(1500, 13)


In [27]:
df.isnull().sum()

annonce_id              0
date_publication       76
titre                   0
ville                   0
quartier              415
type_bien              38
transaction            38
prix                    0
surface                 0
nb_chambres           129
nb_salles_bain        103
etage                 232
annee_construction    203
dtype: int64

In [28]:
df = df.dropna(subset=["date_publication"])
print(df.head())

  annonce_id date_publication                         titre    ville  \
0  ANO000343       2023-05-05        Terrain moderne Meknès   Meknès   
1  ANO001296       2023-08-20       Terrain moderne Kenitra  Kenitra   
2  ANO001212       2023-06-20     Appartement moderne Oujda    Oujda   
3  ANO001000       2024-01-21  Appartement à vente - Tanger   Tanger   
4  ANO001123       2024-03-19              Beau Villa Rabat    Rabat   

   quartier    type_bien transaction       prix  surface  nb_chambres  \
0    Hamria      Terrain       Vente  811401.04   314.40          0.0   
1    Centre      Terrain    Location    3843.55    36.46          0.0   
2  Hay Qods  Appartement    Location    9190.70    95.54          2.0   
3       NaN  Appartement       Vente   30000.00    93.35          1.0   
4   Souissi        Villa    Location   20844.72   264.22          3.0   

   nb_salles_bain  etage  annee_construction  
0             NaN    0.0              2014.0  
1             0.0    NaN          

In [29]:
df["nb_chambres"] = df["nb_chambres"].fillna(df["nb_chambres"].median())
df["nb_salles_bain"] = df["nb_salles_bain"].fillna(df["nb_salles_bain"].median())
df["annee_construction"] = df ["annee_construction"].fillna(df["annee_construction"].median())
df["etage"]= df ["etage"].fillna(0)
print(df.head())


  annonce_id date_publication                         titre    ville  \
0  ANO000343       2023-05-05        Terrain moderne Meknès   Meknès   
1  ANO001296       2023-08-20       Terrain moderne Kenitra  Kenitra   
2  ANO001212       2023-06-20     Appartement moderne Oujda    Oujda   
3  ANO001000       2024-01-21  Appartement à vente - Tanger   Tanger   
4  ANO001123       2024-03-19              Beau Villa Rabat    Rabat   

   quartier    type_bien transaction       prix  surface  nb_chambres  \
0    Hamria      Terrain       Vente  811401.04   314.40          0.0   
1    Centre      Terrain    Location    3843.55    36.46          0.0   
2  Hay Qods  Appartement    Location    9190.70    95.54          2.0   
3       NaN  Appartement       Vente   30000.00    93.35          1.0   
4   Souissi        Villa    Location   20844.72   264.22          3.0   

   nb_salles_bain  etage  annee_construction  
0             1.0    0.0              2014.0  
1             0.0    0.0          

In [30]:
df["prix"] = pd.to_numeric(df["prix"], errors="coerce")
df["surface"] = pd.to_numeric(df["surface"], errors="coerce")
df["nb_chambres"] = pd.to_numeric(df["nb_chambres"], errors="coerce")
df["nb_salles_bain"] = pd.to_numeric(df["nb_salles_bain"], errors="coerce")

df["date_publication"] = pd.to_datetime(df["date_publication"], errors="coerce")
print(df.head())

  annonce_id date_publication                         titre    ville  \
0  ANO000343       2023-05-05        Terrain moderne Meknès   Meknès   
1  ANO001296       2023-08-20       Terrain moderne Kenitra  Kenitra   
2  ANO001212       2023-06-20     Appartement moderne Oujda    Oujda   
3  ANO001000       2024-01-21  Appartement à vente - Tanger   Tanger   
4  ANO001123       2024-03-19              Beau Villa Rabat    Rabat   

   quartier    type_bien transaction       prix  surface  nb_chambres  \
0    Hamria      Terrain       Vente  811401.04   314.40          0.0   
1    Centre      Terrain    Location    3843.55    36.46          0.0   
2  Hay Qods  Appartement    Location    9190.70    95.54          2.0   
3       NaN  Appartement       Vente   30000.00    93.35          1.0   
4   Souissi        Villa    Location   20844.72   264.22          3.0   

   nb_salles_bain  etage  annee_construction  
0             1.0    0.0              2014.0  
1             0.0    0.0          

In [31]:
df["ville"] = df["ville"].str.lower().str.strip()
df["quartier"] = df["quartier"].str.lower().str.strip()
df["type_bien"] = df["type_bien"].str.lower().str.strip()
df["transaction"] = df["transaction"].str.lower().str.strip()
print 

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [32]:
df["prix_m2"] = df["prix"] / df["surface"]
df["prix_m2"]=df["prix_m2"].astype("int")
df.head()

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction,prix_m2
0,ANO000343,2023-05-05,Terrain moderne Meknès,meknès,hamria,terrain,vente,811401.04,314.40,0.0,1.0,0.0,2014.0,2580
1,ANO001296,2023-08-20,Terrain moderne Kenitra,kenitra,centre,terrain,location,3843.55,36.46,0.0,0.0,0.0,2008.0,105
2,ANO001212,2023-06-20,Appartement moderne Oujda,oujda,hay qods,appartement,location,9190.70,95.54,2.0,2.0,0.0,2012.0,96
3,ANO001000,2024-01-21,Appartement à vente - Tanger,tanger,NaN,appartement,vente,30000.00,93.35,1.0,1.0,3.0,2008.0,321
4,ANO001123,2024-03-19,Beau Villa Rabat,rabat,souissi,villa,location,20844.72,264.22,3.0,1.0,0.0,2006.0,78


In [38]:
max_prix = df["prix"].max()
min_prix = df["prix"].min()
print(max_prix)
print(min_prix)

80000000.0
120.0


In [34]:
df.to_sql("clean",engine,if_exists='append',schema='silver', index=False)

ProgrammingError: (psycopg2.errors.InvalidSchemaName) ERREUR:  le schéma « silver » n'existe pas
LINE 2: CREATE TABLE silver.clean (
                     ^

[SQL: 
CREATE TABLE silver.clean (
	annonce_id TEXT, 
	date_publication TIMESTAMP WITHOUT TIME ZONE, 
	titre TEXT, 
	ville TEXT, 
	quartier TEXT, 
	type_bien TEXT, 
	transaction TEXT, 
	prix FLOAT(53), 
	surface FLOAT(53), 
	nb_chambres FLOAT(53), 
	nb_salles_bain FLOAT(53), 
	etage FLOAT(53), 
	annee_construction FLOAT(53), 
	prix_m2 BIGINT
)

]
(Background on this error at: https://sqlalche.me/e/20/f405)